# Talking-avatar prototype: MuseTalk 1.5 (open source, MIT) with the cloned voice
A still photo + the cloned-voice audio from the TTS test become a lip-synced talking video (English, Hindi, Hinglish).

1. **Runtime → Change runtime type → T4 GPU**, Save.
2. Run the cells in order (setup takes 10-15 minutes the first time).
3. When asked, upload together: **one front-facing photo** (your own face: you consent) and the audio files
   `maya2_en.wav`, `maya2_hi.wav`, `maya2_hinglish.wav` from `stt-test/tts-results/out/`.
4. The last cell plays the videos and downloads `avatar-results.zip`: send it back.

Use only a face and a voice whose owner agreed. Mark any shared video as AI-generated. Test only, not for client use.

In [ ]:
!nvidia-smi | head -15
!apt-get -qq install -y ffmpeg > /dev/null && ffmpeg -version | head -1

## Setup: MuseTalk needs Python 3.10 and old PyTorch, so it gets its own environment

In [ ]:
%cd /content
!git clone -q https://github.com/TMElyralab/MuseTalk.git
!pip -q install uv
!uv venv --python 3.10 /content/mt
PIP = '/content/mt/bin/python -m pip'
!uv pip install -q --python /content/mt/bin/python torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
# the inference path does not need tensorflow/tensorboard/gradio (training and demo only): leave them out
!grep -v -i -E "^(tensorflow|tensorboard|gradio)" /content/MuseTalk/requirements.txt > /content/req.txt
!uv pip install -q --python /content/mt/bin/python -r /content/req.txt
!uv pip install -q --python /content/mt/bin/python openmim mmengine
!uv pip install -q --python /content/mt/bin/python "mmcv==2.0.1" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
# mmpose needs chumpy, whose old build script wants pip and setuptools inside the build: build it by hand first
!uv pip install -q --python /content/mt/bin/python pip "setuptools<70" wheel
!uv pip install -q --python /content/mt/bin/python --no-build-isolation "chumpy==0.70"
!uv pip install -q --python /content/mt/bin/python "mmdet==3.1.0" "mmpose==1.1.0"
# fail loudly here instead of printing "ready" and breaking later
!/content/mt/bin/python -c "import torch, mmcv, mmdet, mmpose; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmpose', mmpose.__version__)"
print('environment ready (if the line above shows versions and no error)')

In [ ]:
%cd /content/MuseTalk
# huggingface_hub must stay below 1.0: transformers 4.x refuses 2.x, and 2.x also drops the huggingface-cli command used below
!uv pip install -q --python /content/mt/bin/python "huggingface_hub[cli]==0.25.2" gdown
import os
for d in ['musetalk','musetalkV15','syncnet','dwpose','face-parse-bisent','sd-vae','whisper']: os.makedirs(f'models/{d}', exist_ok=True)
H = '/content/mt/bin/huggingface-cli'
!{H} download TMElyralab/MuseTalk --local-dir models --include "musetalkV15/musetalk.json" "musetalkV15/unet.pth"
!{H} download stabilityai/sd-vae-ft-mse --local-dir models/sd-vae --include "config.json" "diffusion_pytorch_model.bin"
!{H} download openai/whisper-tiny --local-dir models/whisper --include "config.json" "pytorch_model.bin" "preprocessor_config.json"
!{H} download yzd-v/DWPose --local-dir models/dwpose --include "dw-ll_ucoco_384.pth"
!{H} download ByteDance/LatentSync --local-dir models/syncnet --include "latentsync_syncnet.pt"
!/content/mt/bin/gdown --id 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O models/face-parse-bisent/79999_iter.pth
!curl -sL https://download.pytorch.org/models/resnet18-5c106cde.pth -o models/face-parse-bisent/resnet18-5c106cde.pth
print(sorted(os.listdir('models')))

## Inputs: one photo + the three cloned-voice audio files

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
os.makedirs('/content/MuseTalk/data/in', exist_ok=True)
image = [n for n in up if n.lower().endswith(('.jpg', '.jpeg', '.png'))]
wavs = [n for n in up if n.lower().endswith('.wav')]
assert len(image) == 1 and wavs, 'upload exactly one photo and the .wav files'
for n in image + wavs: shutil.move(n, f'/content/MuseTalk/data/in/{n}')
tasks = '\n'.join(f'task_{i}:\n video_path: "data/in/{image[0]}"\n audio_path: "data/in/{w}"\n' for i, w in enumerate(wavs))
open('/content/MuseTalk/configs/inference/avatar.yaml', 'w').write(tasks)
print(tasks)

In [ ]:
%cd /content/MuseTalk
import time; t0 = time.time()
!/content/mt/bin/python -m scripts.inference --inference_config configs/inference/avatar.yaml --result_dir results/avatar \
  --unet_model_path models/musetalkV15/unet.pth --unet_config models/musetalkV15/musetalk.json --version v15 \
  --whisper_dir models/whisper --ffmpeg_path /usr/bin --use_float16
print('seconds for', len(wavs), 'clips:', round(time.time() - t0))

In [ ]:
import glob
from IPython.display import HTML, display
from base64 import b64encode
vids = sorted(glob.glob('/content/MuseTalk/results/avatar/**/*.mp4', recursive=True))
print(vids)
for v in vids:
    print(v.split('/')[-1])
    display(HTML(f'<video width=360 controls src="data:video/mp4;base64,{b64encode(open(v, "rb").read()).decode()}"></video>'))
!cd /content/MuseTalk/results && zip -qr /content/avatar-results.zip avatar
files.download('/content/avatar-results.zip')